# api-sports

In [ ]:
api_key = '9b0c63c7ffb1e43f0d582ce7438f2181'

## Campeonatos

In [ ]:
# import requests
# import pandas as pd

# api_key = api_key
# url = "https://v3.football.api-sports.io/leagues"
# headers = {'x-apisports-key': api_key}

# # Se quiser filtrar apenas ligas do Brasil para ser mais rápido:
# params = {'country': 'Brazil'} 
# # Se quiser TODAS as ligas do mundo, deixe params = {}

# print("Consultando ligas...")
# response = requests.get(url, headers=headers, params=params).json()

# leagues_list = []

# if response.get('response'):
#     for item in response['response']:
#         league = item['league']
#         country = item['country']
        
#         leagues_list.append({
#             'League_ID': league.get('id'),
#             'Nome': league.get('name'),
#             'Tipo': league.get('type'),
#             'Pais': country.get('name'),
#             'Codigo_Pais': country.get('code')
#         })

#     df_leagues = pd.DataFrame(leagues_list)
    
#     # Salvar para conferência
#     df_leagues.to_csv("csv/lista_ligas_brasil.csv", index=False, encoding='utf-8-sig')
#     print(f"Sucesso! {len(df_leagues)} ligas encontradas e salvas em 'lista_ligas_brasil.csv'.")
    
#     # Mostrar as primeiras linhas no console
#     print(df_leagues.head())
# else:
#     print("Erro ou nenhuma liga encontrada:", response.get('errors'))

## Temporadas

In [ ]:
import requests
import pandas as pd

# Configurações
api_key = api_key
url = "https://v3.football.api-sports.io/leagues"
headers = {'x-apisports-key': api_key}
params = {'country': 'Brazil'} 

print("Consultando ligas e temporadas do Brasil...")
response = requests.get(url, headers=headers, params=params).json()

seasons_list = []

if response.get('response'):
    for item in response['response']:
        league = item['league']
        # Cada item de 'response' tem uma lista de 'seasons'
        seasons = item['seasons']
        
        for s in seasons:
            seasons_list.append({
                'League_ID': league.get('id'),
                'Liga_Nome': league.get('name'),
                'Temporada': s.get('year'),
                'Data_Inicio': s.get('start'),
                'Data_Fim': s.get('end'),
                'Atual': s.get('current') # Indica se é a temporada que está ocorrendo agora
            })

    df_seasons = pd.DataFrame(seasons_list)
    
    # Ordenar por liga e temporada mais recente
    df_seasons = df_seasons.sort_values(by=['League_ID', 'Temporada'], ascending=[True, False])
    
    # Salvar
    df_seasons.to_csv("cs/temporadas_brasil.csv", index=False, encoding='utf-8-sig')
    print(f"Sucesso! {len(df_seasons)} temporadas mapeadas.")
    
    # Exemplo: Filtrar apenas as temporadas do Brasileirão Série A (ID 71 geralmente)
    # serie_a = df_seasons[df_seasons['League_ID'] == 71]
    # print(serie_a)
    
    print(df_seasons.head(20))
else:
    print("Erro ou nada encontrado:", response.get('errors'))

## Times

In [ ]:
import requests
import pandas as pd
import time

# Configurações
API_KEY = api_key
SEASON = 2024
HEADERS = {'x-apisports-key': API_KEY}
URL = "https://v3.football.api-sports.io/teams"

# A ORDEM QUE DEFINIMOS
LEAGUES_ORDER = [
    {"id": 475, "nome": "Paulista - A1"},
    {"id": 477, "nome": "Gaúcho - 1"},
    {"id": 606, "nome": "Paranaense - 1"},
    {"id": 624, "nome": "Carioca - 1"},
    {"id": 629, "nome": "Mineiro - 1"}
]

all_teams_data = []

print(f"🏘️ Iniciando busca de times para a Temporada {SEASON}...")

for league in LEAGUES_ORDER:
    league_id = league["id"]
    league_name = league["nome"]
    
    # Parâmetros para trazer todos os times da liga naquela temporada
    params = {
        'league': league_id,
        'season': SEASON
    }
    
    try:
        print(f"🔍 Buscando times de: {league_name}...")
        response = requests.get(URL, headers=HEADERS, params=params).json()
        
        if response.get('errors'):
            print(f"❌ Erro na API: {response['errors']}")
            continue

        teams_list = response.get('response', [])
        
        if not teams_list:
            print(f"⚠️ Nenhum time encontrado para {league_name} em {SEASON}.")
            continue

        for item in teams_list:
            team = item['team']
            venue = item.get('venue', {}) # Informações do estádio
            
            all_teams_data.append({
                'League_ID': league_id,
                'Campeonato': league_name,
                'Time_ID': team.get('id'),
                'Nome_Time': team.get('name'),
                'Fundação': team.get('founded'),
                'Cidade': team.get('city'),
                'Estádio': venue.get('name'),
                'Capacidade_Estádio': venue.get('capacity')
            })
            
        print(f"✅ {len(teams_list)} times encontrados em {league_name}.")
        
        # Pausa para não estourar o limite de requisições por minuto
        time.sleep(2)

    except Exception as e:
        print(f"❌ Erro ao processar liga {league_id}: {e}")

# Gerar CSV final
if all_teams_data:
    df_teams = pd.DataFrame(all_teams_data)
    df_teams.to_csv("csv/times_estaduais_2024.csv", index=False, encoding='utf-8-sig')
    print(f"\n🚀 Sucesso! Total de {len(df_teams)} times extraídos.")
    print(df_teams[['Campeonato', 'Nome_Time', 'Cidade']].head())
else:
    print("\n❌ Nenhum time foi capturado.")

## Jogadores

In [ ]:
import requests
import pandas as pd
import time
import os

# Configurações Iniciais
API_KEY = api_key 
SEASON = 2024
HEADERS = {'x-apisports-key': API_KEY}
URL_TEAMS = "https://v3.football.api-sports.io/teams"
URL_PLAYERS = "https://v3.football.api-sports.io/players"

LEAGUES_ORDER = [
    {"id": 475, "nome": "Paulista - A1"},
    {"id": 477, "nome": "Gaúcho - 1"},
    {"id": 606, "nome": "Paranaense - 1"},
    {"id": 624, "nome": "Carioca - 1"},
    {"id": 629, "nome": "Mineiro - 1"}
]

all_players_data = []

print(f"🚀 Iniciando extração (MODO SEGURO - PLANO FREE) para {SEASON}...")

for league in LEAGUES_ORDER:
    league_id = league["id"]
    league_name = league["nome"]
    
    print(f"\n--- 1. Coletando times de: {league_name} ---")
    
    # Pausa antes de começar cada liga para limpar o contador de rate limit
    time.sleep(7) 
    
    params_teams = {'league': league_id, 'season': SEASON}
    try:
        res_teams = requests.get(URL_TEAMS, headers=HEADERS, params=params_teams).json()
        teams_list = res_teams.get('response', [])
        
        team_ids = [item['team']['id'] for item in teams_list]
        print(f"✅ {len(team_ids)} times encontrados.")

        for team_id in team_ids:
            page = 1
            total_pages = 1
            
            # Enquanto houver páginas e não ultrapassar o limite de 3 do plano Free
            while page <= total_pages and page <= 3:
                params_players = {'team': team_id, 'league': league_id, 'season': SEASON, 'page': page}
                
                # CHAMADA DA API
                response = requests.get(URL_PLAYERS, headers=HEADERS, params=params_players).json()
                
                # Verificação de Erro de Rate Limit
                if 'Too many requests' in str(response.get('errors', '')):
                    print("⏳ Rate limit atingido! Pausando por 30 segundos...")
                    time.sleep(30)
                    continue # Tenta a mesma página novamente

                if response.get('response'):
                    for item in response['response']:
                        p = item['player']
                        # FILTRO SUB-21
                        if p.get('age') and p['age'] <= 21:
                            stat = item['statistics'][0]
                            all_players_data.append({
                                'League_ID': league_id,
                                'Campeonato': league_name,
                                'Time': stat['team']['name'],
                                'Nome': p['name'],
                                'Idade': p['age'],
                                'Posição': stat['games']['position'],
                                'Minutos': stat['games']['minutes'] or 0
                            })
                    
                    total_pages = response.get('paging', {}).get('total', 1)
                    page += 1
                else:
                    break
                
                # PAUSA CRÍTICA: 7 segundos entre cada página/requisição
                print(f"   ∟ Processando página {page-1} do Time ID {team_id}...")
                time.sleep(7) 

    except Exception as e:
        print(f"❌ Falha ao processar {league_name}: {e}")

# Exportação
if all_players_data:
    df = pd.DataFrame(all_players_data)
    df.to_csv(f"csv/sub21_estaduais_{SEASON}.csv", index=False, encoding='utf-8-sig')
    print(f"\n✅ Concluído! {len(df)} jogadores salvos.")
else:
    print("\n❌ Nenhum dado capturado.")